# Xiaohongshu Multimodal Ablation Study — Data Pipeline

This notebook documents the full data construction pipeline:
1. Load the Qilin dataset from HuggingFace
2. Explore schema and field distributions
3. Filter valid image-text notes
4. Coarse-label samples using keyword matching
5. Restrict to selected image parts for tractable download size
6. Balance and export the final dataset (876 samples, ~300 per class)

## 1. Setup

In [2]:
!pip install datasets tqdm pandas -q

In [3]:
import pandas as pd
from collections import Counter
from tqdm import tqdm
from datasets import load_dataset

## 2. Load Qilin Notes Subset

In [4]:
# Load only the 'notes' config from Qilin (1.98M samples)
# Available configs: dqa, notes, recommendation_test/train, search_test/train, user_feat
ds = load_dataset("THUIR/Qilin", "notes", trust_remote_code=True)
notes = ds["train"]
print(ds)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'THUIR/Qilin' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'THUIR/Qilin' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secre

README.md: 0.00B [00:00, ?B/s]

notes/train-00000-of-00005.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

notes/train-00001-of-00005.parquet:   0%|          | 0.00/205M [00:00<?, ?B/s]

notes/train-00002-of-00005.parquet:   0%|          | 0.00/192M [00:00<?, ?B/s]

notes/train-00003-of-00005.parquet:   0%|          | 0.00/191M [00:00<?, ?B/s]

notes/train-00004-of-00005.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1983938 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['note_title', 'note_content', 'note_type', 'video_duration', 'video_height', 'video_width', 'image_num', 'content_length', 'commercial_flag', 'taxonomy1_id', 'taxonomy2_id', 'taxonomy3_id', 'imp_num', 'imp_rec_num', 'imp_search_num', 'click_num', 'click_rec_num', 'click_search_num', 'like_num', 'collect_num', 'comment_num', 'share_num', 'screenshot_num', 'hide_num', 'rec_like_num', 'rec_collect_num', 'rec_comment_num', 'rec_share_num', 'rec_follow_num', 'search_like_num', 'search_collect_num', 'search_comment_num', 'search_share_num', 'search_follow_num', 'accum_like_num', 'accum_collect_num', 'accum_comment_num', 'view_time', 'rec_view_time', 'search_view_time', 'valid_view_times', 'full_view_times', 'note_idx', 'image_path'],
        num_rows: 1983938
    })
})


## 3. Schema Exploration

In [5]:
print("=== Column names ===")
print(notes.column_names)

print("\n=== Total samples ===")
print(len(notes))

print("\n=== First sample ===")
print(notes[0])

=== Column names ===
['note_title', 'note_content', 'note_type', 'video_duration', 'video_height', 'video_width', 'image_num', 'content_length', 'commercial_flag', 'taxonomy1_id', 'taxonomy2_id', 'taxonomy3_id', 'imp_num', 'imp_rec_num', 'imp_search_num', 'click_num', 'click_rec_num', 'click_search_num', 'like_num', 'collect_num', 'comment_num', 'share_num', 'screenshot_num', 'hide_num', 'rec_like_num', 'rec_collect_num', 'rec_comment_num', 'rec_share_num', 'rec_follow_num', 'search_like_num', 'search_collect_num', 'search_comment_num', 'search_share_num', 'search_follow_num', 'accum_like_num', 'accum_collect_num', 'accum_comment_num', 'view_time', 'rec_view_time', 'search_view_time', 'valid_view_times', 'full_view_times', 'note_idx', 'image_path']

=== Total samples ===
1983938

=== First sample ===
{'note_title': '这题我会！每年都要灌几十斤的肉！', 'note_content': '每年只要到灌香肠的环节，就说明快要过年了🧨[偷笑R]有一种让人幸福的仪式感！家里只有我会灌，所以每年都是我弄[笑哭R]还好有老公帮忙切肉，40斤香肠一会就灌好啦～#腊肠[话题]# ', 'note_type': 2.0, 'video_duration': 227.0, 

In [6]:
# Quick distribution analysis on first 5000 rows
sample = notes.select(range(5000))

print("=== note_type distribution ===")
print(Counter(sample["note_type"]))   # 1.0 = image-text, 2.0 = video

print("\n=== commercial_flag distribution ===")
print(Counter(sample["commercial_flag"]))  # 0.0 = normal, 2.0 = commercial

print("\n=== image_num distribution ===")
print(dict(sorted(Counter(sample["image_num"]).items())))

=== note_type distribution ===
Counter({1.0: 4075, 2.0: 925})

=== commercial_flag distribution ===
Counter({0.0: 4921, 2.0: 79})

=== image_num distribution ===
{0.0: 62, 1.0: 1910, 2.0: 495, 3.0: 481, 4.0: 415, 5.0: 376, 6.0: 316, 7.0: 194, 8.0: 145, 9.0: 186, 10.0: 102, 11.0: 49, 12.0: 53, 13.0: 34, 14.0: 24, 15.0: 25, 16.0: 15, 17.0: 29, 18.0: 88, 19.0: 1}


## 4. Keyword-Based Coarse Labelling

Qilin does not provide labels for our 3-class task. We use keyword matching on title + content to assign coarse labels, with a tie-breaking rule (knowledge > food > fashion) to avoid over-labelling ambiguous notes as fashion_beauty.

In [7]:
def coarse_label(title, content):
    text = str(title) + str(content)

    # Fashion/beauty: keep keywords specific to appearance & grooming
    fashion_kw = [
        "穿搭", "ootd", "outfit", "搭配", "妆容", "护肤",
        "发型", "美甲", "口红", "粉底", "眼影", "美妆",
        "彩妆", "skincare", "makeup", "显瘦", "显高",
        "今日穿", "护肤品", "精华", "防晒", "腮红", "眉笔"
    ]

    # Food/travel: broader coverage including dining, venues, and trips
    food_kw = [
        "探店", "餐厅", "咖啡", "旅行", "酒店", "景点", "打卡",
        "好吃", "美食", "旅游", "民宿", "必吃", "必去",
        "好喝", "甜品", "烧烤", "火锅", "自助", "下午茶",
        "测评", "食", "吃", "喝", "店"
    ]

    # Knowledge/tutorial: how-to, guides, tips, experience sharing
    know_kw = [
        "教程", "步骤", "如何", "怎么", "经验", "流程",
        "备考", "干货", "指南", "避雷", "攻略",
        "方法", "技巧", "入门", "新手", "学习", "总结",
        "分享", "必看", "注意", "建议"
    ]

    fashion_score = sum(k in text for k in fashion_kw)
    food_score    = sum(k in text for k in food_kw)
    know_score    = sum(k in text for k in know_kw)

    scores = {
        "fashion_beauty":     fashion_score,
        "food_travel":        food_score,
        "knowledge_tutorial": know_score
    }

    best = max(scores, key=scores.get)
    if scores[best] == 0:
        return "unknown"

    # Tie-breaking: knowledge > food > fashion
    # Prevents generic words (好看, 分享) from pulling ambiguous notes into fashion_beauty
    if scores["knowledge_tutorial"] == scores[best]:
        return "knowledge_tutorial"
    if scores["food_travel"] == scores[best]:
        return "food_travel"
    return best

## 5. Full Dataset Filtering

Filter criteria:
- `note_type == 1.0` — image-text notes only (exclude video)
- `commercial_flag == 0.0` — exclude commercial/sponsored content
- `image_num > 0` and `image_path` not empty — must have at least one image
- keyword label is not 'unknown'

In [8]:
results = []

for row in tqdm(notes, desc="Filtering all 1.98M notes"):
    if row["note_type"]      != 1.0:  continue  # image-text only
    if row["commercial_flag"]!= 0.0:  continue  # exclude ads
    if row["image_num"]      == 0.0:  continue  # must have image
    if not row["image_path"]:         continue  # image_path must not be empty list

    label = coarse_label(row["note_title"], row["note_content"])
    if label == "unknown":            continue

    results.append({
        "note_idx":       row["note_idx"],
        "title":          row["note_title"],
        "content":        row["note_content"],
        "image_path":     row["image_path"][0],   # use cover image only
        "content_length": row["content_length"],
        "label":          label
    })

df_all = pd.DataFrame(results)
print(f"After filtering: {len(df_all)} samples")
print(df_all["label"].value_counts())

Filtering all 1.98M notes: 100%|██████████| 1983938/1983938 [15:56<00:00, 2075.22it/s]


After filtering: 521290 samples
label
knowledge_tutorial    303372
food_travel           180840
fashion_beauty         37078
Name: count, dtype: int64


## 6. Restrict to Selected Image Parts

Images are distributed across 118 tar.gz archives about 340 GB total. To make the image download tractable, we identify which parts contain the most balanced samples across all three classes, then restrict sampling to those parts only around 12 GB to download.

In [9]:
# Add part column to identify which archive each image belongs to
df_all["part"] = df_all["image_path"].apply(lambda x: x.split("/")[1])

# Show top 20 parts by total sample count
part_counts = df_all.groupby(["part", "label"]).size().unstack(fill_value=0)
part_counts["total"] = part_counts.sum(axis=1)
part_counts = part_counts.sort_values("total", ascending=False)

print("Top 20 parts by sample count:")
print(part_counts.head(20))

Top 20 parts by sample count:
label     fashion_beauty  food_travel  knowledge_tutorial  total
part                                                            
part_113             219         1611                4348   6178
part_114             268         1542                3647   5457
part_115             251         1533                3621   5405
part_116             262         1545                3563   5370
part_112             287         1568                3389   5244
part_117             209         1461                3413   5083
part_13              345         1594                2594   4533
part_38              380         1596                2555   4531
part_20              319         1582                2607   4508
part_14              336         1571                2573   4480
part_33              339         1616                2522   4477
part_37              323         1568                2578   4469
part_6               323         1552                2592   

In [10]:
# Select top 4 parts — all three classes well-represented, only ~12 GB to download
# part_113: 5785 samples, part_114: 5085, part_115: 5015, part_116: 5038
selected_parts = ["part_113", "part_114", "part_115", "part_116"]

results_selected = []
for row in df_all.itertuples():
    if row.part not in selected_parts:
        continue
    label = coarse_label(row.title, row.content)
    if label == "unknown":
        continue
    results_selected.append({
        "note_idx":       row.note_idx,
        "title":          row.title,
        "content":        row.content,
        "image_path":     row.image_path,
        "content_length": row.content_length,
        "label":          label
    })

df_selected = pd.DataFrame(results_selected)
print("Distribution within selected parts:")
print(df_selected["label"].value_counts())

Distribution within selected parts:
label
knowledge_tutorial    15179
food_travel            6231
fashion_beauty         1000
Name: count, dtype: int64


## 7. Balanced Sampling & Quality Check

In [11]:
# Sample 300 per class (random_state=42 for reproducibility)
df_final = (
    df_selected
    .groupby("label", group_keys=False)
    .apply(lambda x: x.sample(n=300, random_state=42))
    .reset_index(drop=True)
)

print(f"Final dataset: {len(df_final)} samples")
print(df_final["label"].value_counts())

Final dataset: 900 samples
label
fashion_beauty        300
food_travel           300
knowledge_tutorial    300
Name: count, dtype: int64


/tmp/ipykernel_25491/302940659.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=300, random_state=42))


In [12]:
# Spot-check: sample 3 titles per class to verify label quality
for label in ["fashion_beauty", "food_travel", "knowledge_tutorial"]:
    print(f"\n--- {label} ---")
    for title in df_final[df_final["label"] == label]["title"].sample(3, random_state=99):
        print(f"  {title}")


--- fashion_beauty ---
  在徐州｜冬日的古柏广场🌲
  秋冬lulu真绝绝子！
  nan

--- food_travel ---
  剧本｜寒门·贰选角推荐及部分测评
  东北地图
  纯正俄罗斯进口奶酪！真便宜呀！

--- knowledge_tutorial ---
  周岁礼KT 板上的生日日期怎么写？？
  装修智商税之一——瓷砖
  以感恩为园所文化的幼儿园


In [13]:
# Remove rows with missing titles
df_final = df_final[
    df_final["title"].notna() & (df_final["title"].astype(str).str.strip() != "nan")
].reset_index(drop=True)

print(f"After cleaning: {len(df_final)} samples")
print(df_final["label"].value_counts())

After cleaning: 876 samples
label
fashion_beauty        293
knowledge_tutorial    292
food_travel           291
Name: count, dtype: int64


## 8. Export

Save as `metadata_candidates.csv` for manual review.
After human verification, the final file will be renamed to `metadata.csv`.

In [14]:
df_final.drop(columns=["part"], errors="ignore").to_csv(
    "metadata_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)
print("Saved: metadata_candidates.csv")
print(f"Shape: {df_final.shape}")

Saved: metadata_candidates.csv
Shape: (876, 6)
